# Badge 4: Data Lake Workshop: Lesson 3: Leaving the Data Where it Lands

File formats and views over the non-loaded metadata files.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE4_DLKW_ADMIN` (ACCOUNTADMIN), `BADGE4_DLKW_ROLE` (SYSADMIN) and `BADGE4_DLKW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.BADGE4_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE4_DLKW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE4_DLKW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE4_DLKW_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
use schema ZENAS_ATHLEISURE_DB.PRODUCTS;
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: 🥋 Leaving Data Where It Lands!

In [ ]:
%%sql -r list_default_ctx
list @product_metadata;

In [ ]:
%%sql -r list_fully_qualified
list @zenas_athleisure_db.products.product_metadata;

In [ ]:
%%sql -r select_dollar_1
select $1
from @product_metadata; 

## Page: 🥋 Helping Snowflake Interpret Data Files

In [ ]:
%%sql -r select_one_file
select $1
from @product_metadata/product_coordination_suggestions.txt; 

## Page: 🥋 Querying with the Help of File Formats

In [ ]:
%%sql -r zmd_file_format_1_create
create file format zmd_file_format_1
RECORD_DELIMITER = '^';

In [ ]:
%%sql -r zmd_file_format_1_select
select $1
from @product_metadata/product_coordination_suggestions.txt
(file_format => zmd_file_format_1);

In [ ]:
%%sql -r zmd_file_format_2_create
create file format zmd_file_format_2
FIELD_DELIMITER = '^';

In [ ]:
%%sql -r zmd_file_format_2_select
select $1
from @product_metadata/product_coordination_suggestions.txt
(file_format => zmd_file_format_2);

> **Course:** 🥋 A Third Possibility? Replace the question marks with the real delimiters. The block below is the course's template, run as written (the `?` are placeholders).

> ⚠️ **Audit note:** The course's template sets both delimiters to `?`, which Snowflake rejects (the same character cannot be both). It runs as given and reports the error; the next page gives the real delimiters.

In [ ]:
%%sql -r zmd_file_format_3_template
execute immediate 'begin
  execute immediate ''create or replace file format zmd_file_format_3
FIELD_DELIMITER = ''''?''''
RECORD_DELIMITER = ''''?'''''';
  return ''ran without error'';
exception when other then
  return ''Error (as the course predicts): '' || sqlerrm;
end;';

## Page: 🥋 A Solid File Format for the Product Coordination File

In [ ]:
%%sql -r zmd_file_format_3
create or replace file format zmd_file_format_3
FIELD_DELIMITER = '='
RECORD_DELIMITER = '^';

In [ ]:
%%sql -r zmd_file_format_3_select
-- DLKW_020: caret = row separator, equal sign = column separator
select $1, $2
from @product_metadata/product_coordination_suggestions.txt
(file_format => zmd_file_format_3);

## Page: 🎯 Repurpose File Format 1 for the Sweatsuit Sizes File

> **Course:** 🎯 Rewrite zmd_file_format_1 to parse sweatsuit_sizes.txt (DLKW_021, 022): `create or replace`, then run the select.

In [ ]:
%%sql -r zmd_file_format_1_sizes
-- Derived solution: the sizes are separated by semicolons
create or replace file format zmd_file_format_1
RECORD_DELIMITER = ';';

In [ ]:
%%sql -r sizes_select
select $1 as sizes_available
from @product_metadata/sweatsuit_sizes.txt
(file_format => zmd_file_format_1 );

## Page: 🎯 Repurpose File Format 2 for the Sweatband Product Line File

> **Course:** 🎯 Rewrite zmd_file_format_2 to parse swt_product_line.txt: 3 columns and 10 rows (DLKW_023, 024). Then add TRIM_SPACE = TRUE and re-run the select (DLKW_025).

In [ ]:
%%sql -r zmd_file_format_2_product_line
-- Derived solution: records end with ';', fields are separated by '|'
create or replace file format zmd_file_format_2
FIELD_DELIMITER = '|'
RECORD_DELIMITER = ';';

In [ ]:
%%sql -r product_line_select
select $1, $2, $3
from @product_metadata/swt_product_line.txt
(file_format => zmd_file_format_2);

In [ ]:
%%sql -r zmd_file_format_2_trim
-- Derived solution: the course's TRIM_SPACE property
create or replace file format zmd_file_format_2
FIELD_DELIMITER = '|'
RECORD_DELIMITER = ';'
TRIM_SPACE = TRUE;

In [ ]:
%%sql -r product_line_select_trim
select $1, $2, $3
from @product_metadata/swt_product_line.txt
(file_format => zmd_file_format_2);

## Page: 🥋 Make the Sweatsuit Sizes Data Look Great!

> **Course:** 🎯 Make Sure All 3 File Formats Have a Trim Space Property ([TRIM_SPACE = True]).

In [ ]:
%%sql -r trim_all_file_formats
-- Derived solution
alter file format zmd_file_format_1 set TRIM_SPACE = TRUE;
alter file format zmd_file_format_2 set TRIM_SPACE = TRUE;
alter file format zmd_file_format_3 set TRIM_SPACE = TRUE;

> **Course:** 🥋 Dealing with Unexpected Characters: REPLACE($1, chr(13)||chr(10)) removes CRLF (DLKW_027 shows the select).

In [ ]:
%%sql -r sizes_without_crlf
-- From screenshot DLKW_027
select REPLACE($1,chr(13)||chr(10)) as sizes_available
from @product_metadata/sweatsuit_sizes.txt
(file_format => zmd_file_format_1);

## Page: 🥋 Make the Sweatsuit Sizes Data Even Better!

In [ ]:
%%sql -r sizes_nonempty
-- From screenshot DLKW_028: nix the empty 19th row
select REPLACE($1,chr(13)||chr(10)) as sizes_available
from @product_metadata/sweatsuit_sizes.txt
(file_format => zmd_file_format_1)
where sizes_available <> '';

> **Course:** 🥋 Convert Your Select to a View: add this line above your select.

In [ ]:
%%sql -r create_sweatsuit_sizes
create view zenas_athleisure_db.products.sweatsuit_sizes as 
select REPLACE($1,chr(13)||chr(10)) as sizes_available
from @product_metadata/sweatsuit_sizes.txt
(file_format => zmd_file_format_1)
where sizes_available <> '';

In [ ]:
%%sql -r check_sweatsuit_sizes
select * from zenas_athleisure_db.products.sweatsuit_sizes;

## Page: 🎯 Make the Sweatband Product Line File Look Great!

In [ ]:
%%sql -r create_sweatband_product_line
-- Derived solution (column names from DLKW_030)
create view zenas_athleisure_db.products.SWEATBAND_PRODUCT_LINE as
select REPLACE($1,chr(13)||chr(10)) as product_code
,REPLACE($2,chr(13)||chr(10)) as headband_description
,REPLACE($3,chr(13)||chr(10)) as wristband_description
from @product_metadata/swt_product_line.txt
(file_format => zmd_file_format_2)
where product_code <> '';

## Page: 🎯 Make the Product Coordination Data Look Great!

In [ ]:
%%sql -r create_sweatband_coordination
-- Derived solution (column names from DLKW_033)
create view zenas_athleisure_db.products.SWEATBAND_COORDINATION as
select REPLACE($1,chr(13)||chr(10)) as product_code
,REPLACE($2,chr(13)||chr(10)) as has_matching_sweatsuit
from @product_metadata/product_coordination_suggestions.txt
(file_format => zmd_file_format_3)
where product_code <> '';

## Page: 🤖 DORA DLKW02

### 🤖 DORA check DLKW02 goes here
The course has the learner run **DLKW02** (views have no CRLFs) at this point.
It is in `dora/dora_tests.ipynb`, section **DLKW02**, as a local check plus the exact DORA code.

## Page: 🏁 Summary & Next Steps

In [ ]:
%%sql -r summary_1
select product_code, has_matching_sweatsuit
from zenas_athleisure_db.products.sweatband_coordination;

In [ ]:
%%sql -r summary_2
select product_code, headband_description, wristband_description
from zenas_athleisure_db.products.sweatband_product_line;

In [ ]:
%%sql -r summary_3
select sizes_available
from zenas_athleisure_db.products.sweatsuit_sizes;

In [ ]:
%%sql -r summary_counts
-- 10, 10 and 18 rows expected
select (select count(*) from zenas_athleisure_db.products.sweatband_coordination) as coordination_rows,
       (select count(*) from zenas_athleisure_db.products.sweatband_product_line) as product_line_rows,
       (select count(*) from zenas_athleisure_db.products.sweatsuit_sizes) as sizes_rows;